<a href="https://colab.research.google.com/github/DBLS13/Asia-Markets-Structuring-Lab/blob/main/notebooks/asia_markets_structuring_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Asia Markets Structuring Lab
## Dual Currency Investment & Equity-Linked Reverse Convertible

### Scope

This project replicates how a Hong Kong Global Markets structuring / sales
desk turns a client objective into a term sheet, prices the embedded option,
and measures dealer hedge slippage.

### Products

1. **6-Month USD/CNH Dual Currency Investment (DCI)** — a structured FX
   deposit embedding a short FX put. The "enhanced yield" is the put premium.
2. **6-Month HSCEI Equity-Linked Note (ELN)** — a structured deposit
   embedding a short equity put. The coupon above risk-free is the put premium,
   not free yield.

### What this is not

This is not a term sheet, not investment advice, and not a production pricing
tool. It is a personal project that demonstrates structured-product workflow:
client brief → payoff decomposition → pricing → greeks → hedge residual →
suitability.

### Explicit model limitations

| Omission | Why it matters | Why it is acceptable here |
|----------|---------------|--------------------------|
| Volatility smile / skew | Misprices OTM puts; real desks use local-vol or SABR | Flat-vol isolates the core economics cleanly |
| Issuer credit risk | Real notes embed issuer default; affects note price | Adding CVA requires CDS curves beyond scope |
| Funding spread | Desk funds at SOFR + spread, not risk-free | One optional 50 bp add-on is shown as a hook |
| Barriers / autocall | Real ELNs often have knock-in puts | Vanilla put keeps the decomposition transparent |
| Stochastic rates | USD and CNH rates are deterministic here | Short tenor (6M) makes rate vol second-order |
| Transaction taxes / stamp duty | HK stamp duty on equities is 10 bps | Captured in the transaction-cost parameter |

In [1]:
# =============================================================================
# ENVIRONMENT SETUP
# =============================================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", message="datetime.datetime.utcnow",
                        category=DeprecationWarning)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Environment ready.")

Environment ready.


## Market Snapshot — Desk Ticket

All inputs are frozen from a single dated observation (Q2 quarter-end).
No live feeds. Values sourced from public feeds and cross-checked against
typical ranges. For production use, replace with Bloomberg closing data.

| Field | Value | Source / Note |
|-------|-------|---------------|
| Observation date | 2026-06-30 | Q2 quarter-end snapshot |
| **DCI (FX)** | | |
| USD/CNH spot | 7.2450 | Offshore CNH mid |
| USD 6M depo rate | 4.05% Act/365 | SOFR term + spread |
| CNH 6M depo rate | 1.95% Act/365 | CNH HIBOR fixing |
| USD/CNH 6M ATM implied vol | 5.50% | USDCNH BGN V6M |
| **ELN (Equity)** | | |
| HSCEI spot | 7,285 | Hang Seng China Enterprises Index |
| HKD risk-free rate (6M) | 3.85% Act/365 | HONIA OIS |
| HSCEI dividend yield (est.) | 3.40% | Trailing 12M |
| HSCEI 6M ATM implied vol | 24.80% | HSCEI V6M |
| **Common** | | |
| Tenor | 6 months (0.50 yr) | |
| Notional | USD 1,000,000 (DCI) / HKD 1,000,000 (ELN) | |
| Day count | Act/365 for simplicity | |

Data file: `data/market_snapshot_20260630.csv`

In [2]:
# =============================================================================
# FROZEN MARKET SNAPSHOT — 30 JUN 2026 (Q2 QUARTER-END)
# =============================================================================
# This replaces live yfinance calls. Treat as a desk ticket.
# Values sourced from public feeds; replace with BBG closes for production.

# --- DCI (FX) parameters ---
DCI_SPOT = 7.2450             # USD/CNH spot
DCI_USD_RATE = 0.0405         # USD 6M deposit rate
DCI_CNH_RATE = 0.0195         # CNH 6M deposit rate
DCI_VOL = 0.0550              # USD/CNH 6M ATM implied vol
DCI_TENOR = 0.50              # 6 months
DCI_NOTIONAL_USD = 1_000_000  # Notional in USD

# DCI strike: ATM spot (client receives CNH at spot if put exercised)
DCI_STRIKE = DCI_SPOT

# --- ELN (Equity) parameters ---
ELN_SPOT = 7285.00            # HSCEI level
ELN_RATE = 0.0385             # HKD 6M risk-free (HONIA OIS)
ELN_DIV_YIELD = 0.0340        # HSCEI estimated dividend yield
ELN_VOL = 0.2480              # HSCEI 6M ATM implied vol
ELN_TENOR = 0.50              # 6 months
ELN_NOTIONAL_HKD = 1_000_000  # Notional in HKD
ELN_STRIKE_PCT = 0.90         # 90% of spot (10% OTM put)
ELN_STRIKE = ELN_SPOT * ELN_STRIKE_PCT

# --- Simulation parameters ---
TRADING_DAYS_PER_YEAR = 252

# --- Input validation ---
for name, val in [("DCI_SPOT", DCI_SPOT), ("DCI_VOL", DCI_VOL),
                  ("ELN_SPOT", ELN_SPOT), ("ELN_VOL", ELN_VOL)]:
    assert val > 0, f"{name} must be positive"
assert DCI_TENOR > 0 and ELN_TENOR > 0, "Tenor must be positive"

print("Market snapshot loaded — 30 Jun 2026 (Q2 quarter-end)")
print(f"  DCI: USD/CNH {DCI_SPOT:.4f}, vol {DCI_VOL:.2%}, tenor {DCI_TENOR}y")
print(f"  ELN: HSCEI {ELN_SPOT:,.2f}, vol {ELN_VOL:.2%}, "
      f"strike {ELN_STRIKE_PCT:.0%} = {ELN_STRIKE:,.2f}")

Market snapshot loaded — 30 Jun 2026 (Q2 quarter-end)
  DCI: USD/CNH 7.2450, vol 5.50%, tenor 0.5y
  ELN: HSCEI 7,285.00, vol 24.80%, strike 90% = 6,556.50


In [3]:
# =============================================================================
# BLACK-SCHOLES PRICING AND GREEKS ENGINE
# =============================================================================

def _to_array(value, name, positive=False):
    """Convert scalar or array to finite numpy array with optional positivity check."""
    a = np.asarray(value, dtype=float)
    if not np.all(np.isfinite(a)):
        raise ValueError(f"{name} contains non-finite values.")
    if positive and np.any(a <= 0):
        raise ValueError(f"{name} must be strictly positive.")
    return a


def bs_d1_d2(S, K, T, r, sigma, q=0.0):
    """Compute Black-Scholes d1 and d2."""
    S, K, T, sigma = [_to_array(x, n, positive=True)
                      for x, n in [(S,"S"),(K,"K"),(T,"T"),(sigma,"σ")]]
    r, q = _to_array(r, "r"), _to_array(q, "q")
    sqrt_T = np.sqrt(T)
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * sqrt_T)
    d2 = d1 - sigma * sqrt_T
    return d1, d2


def bs_price(S, K, T, r, sigma, opt="call", q=0.0):
    """European option price under Black-Scholes-Merton."""
    opt = opt.lower().strip()
    assert opt in ("call", "put"), "opt must be 'call' or 'put'"
    d1, d2 = bs_d1_d2(S, K, T, r, sigma, q)
    disc_S = S * np.exp(-q * T)
    disc_K = K * np.exp(-r * T)
    if opt == "call":
        return disc_S * norm.cdf(d1) - disc_K * norm.cdf(d2)
    return disc_K * norm.cdf(-d2) - disc_S * norm.cdf(-d1)


def bs_greeks(S, K, T, r, sigma, opt="call", q=0.0):
    """Delta, Gamma, Vega (per 1 vol pt), Theta (per day) under BSM."""
    opt = opt.lower().strip()
    assert opt in ("call", "put")
    d1, d2 = bs_d1_d2(S, K, T, r, sigma, q)
    sqrt_T = np.sqrt(T)
    n_d1 = norm.pdf(d1)
    disc_S = S * np.exp(-q * T)
    disc_K = K * np.exp(-r * T)

    gamma = np.exp(-q * T) * n_d1 / (S * sigma * sqrt_T)
    vega_1pct = disc_S * n_d1 * sqrt_T * 0.01

    if opt == "call":
        delta = np.exp(-q * T) * norm.cdf(d1)
        theta_yr = (-disc_S * n_d1 * sigma / (2 * sqrt_T)
                     - r * disc_K * norm.cdf(d2)
                     + q * disc_S * norm.cdf(d1))
    else:
        delta = np.exp(-q * T) * (norm.cdf(d1) - 1)
        theta_yr = (-disc_S * n_d1 * sigma / (2 * sqrt_T)
                     + r * disc_K * norm.cdf(-d2)
                     - q * disc_S * norm.cdf(-d1))

    return {"delta": delta, "gamma": gamma,
            "vega_1pct": vega_1pct, "theta_day": theta_yr / 365}


print("Black-Scholes engine loaded.")

Black-Scholes engine loaded.


In [4]:
# =============================================================================
# SANITY CHECKS
# =============================================================================

def run_sanity_checks():
    """Put-call parity, non-negativity, delta bounds."""

    # --- Test on DCI FX parameters ---
    c = float(bs_price(DCI_SPOT, DCI_STRIKE, DCI_TENOR, DCI_USD_RATE,
                       DCI_VOL, "call", DCI_CNH_RATE))
    p = float(bs_price(DCI_SPOT, DCI_STRIKE, DCI_TENOR, DCI_USD_RATE,
                       DCI_VOL, "put", DCI_CNH_RATE))
    parity_lhs = c - p
    parity_rhs = (DCI_SPOT * np.exp(-DCI_CNH_RATE * DCI_TENOR)
                  - DCI_STRIKE * np.exp(-DCI_USD_RATE * DCI_TENOR))
    parity_err = abs(parity_lhs - parity_rhs)

    assert c >= 0, "Call price negative"
    assert p >= 0, "Put price negative"
    assert parity_err < 1e-10, f"Put-call parity failed: err={parity_err:.2e}"

    # --- Test on ELN equity parameters ---
    c2 = float(bs_price(ELN_SPOT, ELN_STRIKE, ELN_TENOR, ELN_RATE,
                        ELN_VOL, "call", ELN_DIV_YIELD))
    p2 = float(bs_price(ELN_SPOT, ELN_STRIKE, ELN_TENOR, ELN_RATE,
                        ELN_VOL, "put", ELN_DIV_YIELD))
    parity_err2 = abs((c2 - p2) -
                      (ELN_SPOT * np.exp(-ELN_DIV_YIELD * ELN_TENOR)
                       - ELN_STRIKE * np.exp(-ELN_RATE * ELN_TENOR)))
    assert c2 >= 0 and p2 >= 0
    assert parity_err2 < 1e-10, f"ELN put-call parity failed: {parity_err2:.2e}"

    # --- Delta bounds ---
    d_call = float(bs_greeks(DCI_SPOT, DCI_STRIKE, DCI_TENOR, DCI_USD_RATE,
                             DCI_VOL, "call", DCI_CNH_RATE)["delta"])
    d_put = float(bs_greeks(ELN_SPOT, ELN_STRIKE, ELN_TENOR, ELN_RATE,
                            ELN_VOL, "put", ELN_DIV_YIELD)["delta"])
    assert 0 <= d_call <= 1, f"Call delta out of [0,1]: {d_call}"
    assert -1 <= d_put <= 0, f"Put delta out of [-1,0]: {d_put}"

    print("All sanity checks passed.")
    print(f"  DCI put-call parity error: {parity_err:.2e}")
    print(f"  ELN put-call parity error: {parity_err2:.2e}")
    print(f"  DCI FX call price: {c:.6f} per 1 USD/CNH")
    print(f"  DCI FX put price:  {p:.6f} per 1 USD/CNH")
    print(f"  ELN HSCEI put price: {p2:,.2f} index pts")

run_sanity_checks()

All sanity checks passed.
  DCI put-call parity error: 8.88e-16
  ELN put-call parity error: 6.82e-13
  DCI FX call price: 0.152210 per 1 USD/CNH
  DCI FX put price:  0.077269 per 1 USD/CNH
  ELN HSCEI put price: 195.93 index pts


---

## Product 1: 6-Month USD/CNH Dual Currency Investment

### What it is:

The client deposits USD for 6 months and receives an enhanced coupon above
the USD deposit rate. In exchange, the client sells a USD put / CNH call to
the bank. If USD/CNH fixes below the strike at maturity, the principal is
returned in CNH at the strike rate instead of in USD.

### Decomposition

$$\text{DCI coupon} = \text{USD risk-free deposit rate} + \underbrace{\text{FX put premium (annualised)}}_{\text{compensation for conversion risk}}$$

The "extra yield" is **not** free. It is the annualised premium of the
embedded FX put the client has sold.

### Client profile

HK corporate treasury with USD receivables and CNH payables.
They are comfortable receiving CNH at a rate close to spot because they
have natural CNH liabilities. The DCI is a **yield-enhanced hedging tool**,
not a speculative bet.

In [5]:
# =============================================================================
# PRODUCT 1: DUAL CURRENCY INVESTMENT (USD/CNH)
# =============================================================================

# --- Embedded option: client is SHORT a USD put / CNH call ---
# Garman-Kohlhagen: domestic rate = USD rate, foreign rate = CNH rate
# The client sells a put on USD/CNH (equivalently, a call on CNH/USD)

dci_put_price = float(bs_price(DCI_SPOT, DCI_STRIKE, DCI_TENOR,
                               DCI_USD_RATE, DCI_VOL, "put", DCI_CNH_RATE))

# Annualised premium as a yield on notional
dci_option_yield = (dci_put_price / DCI_SPOT) / DCI_TENOR
dci_total_coupon = DCI_USD_RATE + dci_option_yield
dci_risk_free_return = DCI_NOTIONAL_USD * DCI_USD_RATE * DCI_TENOR
dci_enhanced_return = DCI_NOTIONAL_USD * dci_total_coupon * DCI_TENOR

print("=" * 60)
print("DCI INDICATIVE TERM SHEET")
print("=" * 60)
print(f"  Underlying:           USD/CNH")
print(f"  Spot:                 {DCI_SPOT:.4f}")
print(f"  Strike:               {DCI_STRIKE:.4f} (ATM)")
print(f"  Tenor:                {DCI_TENOR:.2f}y (6 months)")
print(f"  USD deposit rate:     {DCI_USD_RATE:.2%}")
print(f"  CNH deposit rate:     {DCI_CNH_RATE:.2%}")
print(f"  Implied vol:          {DCI_VOL:.2%}")
print(f"  Embedded put value:   {dci_put_price:.6f} per 1 USD/CNH")
print(f"  Option yield (ann.):  {dci_option_yield:.2%}")
print(f"  Total DCI coupon:     {dci_total_coupon:.2%} p.a.")
print(f"  Risk-free 6M return:  USD {dci_risk_free_return:,.2f}")
print(f"  DCI 6M return:        USD {dci_enhanced_return:,.2f}")
print(f"  Yield pickup:         {dci_option_yield:.2%} p.a.")
print()

# --- Dealer greeks on the embedded put (dealer is LONG the put) ---
dci_greeks = bs_greeks(DCI_SPOT, DCI_STRIKE, DCI_TENOR,
                       DCI_USD_RATE, DCI_VOL, "put", DCI_CNH_RATE)

print("DEALER GREEKS (long put, per 1 USD/CNH notional):")
for k, v in dci_greeks.items():
    print(f"  {k:12s}: {float(v):+.6f}")

# Scale to notional (in units of USD/CNH)
dci_notional_units = DCI_NOTIONAL_USD  # each unit = 1 USD
print(f"\nSCALED TO USD {DCI_NOTIONAL_USD:,.0f} NOTIONAL:")
print(f"  Delta (USD): {float(dci_greeks['delta']) * dci_notional_units:+,.0f}")
print(f"  Vega (1 vol pt): {float(dci_greeks['vega_1pct']) * dci_notional_units:+,.2f}")
print(f"  Theta (per day): {float(dci_greeks['theta_day']) * dci_notional_units:+,.2f}")

# --- Terminal payoff diagram ---
spot_grid = np.linspace(DCI_SPOT * 0.90, DCI_SPOT * 1.10, 500)

# Excess vs risk-free = coupon pickup if no conversion,
#                       coupon pickup - FX loss if converted
dci_excess = np.where(
    spot_grid >= DCI_STRIKE,
    DCI_NOTIONAL_USD * dci_option_yield * DCI_TENOR,  # keep coupon
    DCI_NOTIONAL_USD * (dci_option_yield * DCI_TENOR
                        - (1 - spot_grid / DCI_STRIKE))  # coupon - FX loss
)

# Break-even: spot where excess = 0
dci_breakeven = DCI_STRIKE * (1 - dci_option_yield * DCI_TENOR)

# --- DCI client payoff chart ---
fig_dci = go.Figure()

fig_dci.add_trace(
    go.Scatter(
        x=spot_grid,
        y=dci_excess,
        mode="lines",
        name="DCI excess vs USD deposit",
        line=dict(width=3, color="#1f77b4"),
    )
)

# Reference line: no excess return versus plain USD deposit
fig_dci.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray",
    line_width=1,
)

# Vertical reference lines
fig_dci.add_vline(
    x=dci_breakeven,
    line_dash="dot",
    line_color="red",
    line_width=2,
)

fig_dci.add_vline(
    x=DCI_STRIKE,
    line_dash="dot",
    line_color="black",
    line_width=2,
)

# Labels placed at different vertical positions to avoid overlap
fig_dci.add_annotation(
    x=dci_breakeven,
    y=0.96,
    xref="x",
    yref="paper",
    text=f"Break-even<br>{dci_breakeven:.4f}",
    showarrow=False,
    xanchor="right",
    xshift=-8,
    font=dict(color="red", size=13),
    bgcolor="rgba(255,255,255,0.85)",
)

fig_dci.add_annotation(
    x=DCI_STRIKE,
    y=0.84,
    xref="x",
    yref="paper",
    text=f"Strike<br>{DCI_STRIKE:.4f}",
    showarrow=False,
    xanchor="left",
    xshift=8,
    font=dict(color="black", size=13),
    bgcolor="rgba(255,255,255,0.85)",
)

fig_dci.update_layout(
    title="6M USD/CNH DCI — Client Excess Return vs USD Deposit",
    xaxis_title="USD/CNH at Maturity",
    yaxis_title="Excess Return (USD)",
    template="plotly_white",
    height=550,
    margin=dict(t=100, l=90, r=50, b=80),
    showlegend=False,
)

fig_dci.show()

# --- Spot × Vol scenario grid ---
spot_shocks = np.linspace(-10, 10, 21) / 100  # ±10%
vol_shocks = np.linspace(-3, 3, 13) / 100     # ±3 vol pts

spot_scenarios = DCI_SPOT * (1 + spot_shocks)
vol_scenarios = DCI_VOL + vol_shocks

S_mesh, V_mesh = np.meshgrid(spot_scenarios, vol_scenarios)

# MTM of dealer's long put after 3 months (T remaining = 0.25)
T_remaining = DCI_TENOR / 2
mtm_prices = bs_price(S_mesh, DCI_STRIKE, T_remaining,
                      DCI_USD_RATE, V_mesh, "put", DCI_CNH_RATE)
initial_put_price = dci_put_price
dci_mtm_pnl = (mtm_prices - initial_put_price) * DCI_NOTIONAL_USD

fig_dci_grid = go.Figure(data=go.Heatmap(
    x=np.round(spot_scenarios, 4),
    y=np.round(vol_scenarios * 100, 1),
    z=dci_mtm_pnl,
    colorscale="RdYlGn", zmid=0,
    colorbar=dict(title="MTM P&L (USD)")))
fig_dci_grid.update_layout(
    title="DCI Dealer Spot × Vol MTM P&L (3M into trade, USD)",
    xaxis_title="USD/CNH Spot",
    yaxis_title="Implied Vol (%)",
    template="plotly_white", height=550)
fig_dci_grid.show()

print(f"\nDCI break-even USD/CNH: {dci_breakeven:.4f}")
print(f"Client max loss: full conversion to CNH at strike "
      f"({DCI_STRIKE:.4f}) if spot → 0 (theoretical)")


DCI INDICATIVE TERM SHEET
  Underlying:           USD/CNH
  Spot:                 7.2450
  Strike:               7.2450 (ATM)
  Tenor:                0.50y (6 months)
  USD deposit rate:     4.05%
  CNH deposit rate:     1.95%
  Implied vol:          5.50%
  Embedded put value:   0.077269 per 1 USD/CNH
  Option yield (ann.):  2.13%
  Total DCI coupon:     6.18% p.a.
  Risk-free 6M return:  USD 20,250.00
  DCI 6M return:        USD 30,915.20
  Yield pickup:         2.13% p.a.

DEALER GREEKS (long put, per 1 USD/CNH notional):
  delta       : -0.382379
  gamma       : +1.344618
  vega_1pct   : +0.019409
  theta_day   : -0.000125

SCALED TO USD 1,000,000 NOTIONAL:
  Delta (USD): -382,379
  Vega (1 vol pt): +19,409.24
  Theta (per day): -124.51



DCI break-even USD/CNH: 7.1677
Client max loss: full conversion to CNH at strike (7.2450) if spot → 0 (theoretical)


---

## Product 2: 6-Month HSCEI Equity-Linked Note

### What it is

The client deposits HKD for 6 months and receives a coupon above the
HKD risk-free rate. In exchange, the client sells a put on HSCEI to the
bank, struck at 90% of spot. If HSCEI closes below the strike at maturity,
the client suffers an equity loss proportional to the decline below the
strike.

### Decomposition

$$\text{ELN coupon} = \text{HKD risk-free rate} + \underbrace{\text{Put premium (annualised)}}_{\text{compensation for equity downside}}$$

The coupon is the **future value of the put the client sold**. If HSCEI
is unchanged, the client keeps the coupon. If it sells off through the
strike, the client owns the equity downside.

### Client profile

Private-bank client seeking 8–12% p.a. indicative coupon, willing to accept
equity downside below a 10% buffer. Suitable for a moderately bullish or
range-bound view on HK/China equities.


In [6]:
# =============================================================================
# PRODUCT 2: EQUITY-LINKED NOTE (HSCEI)
# =============================================================================

# --- Embedded option: client is SHORT a put on HSCEI ---
eln_put_price = float(bs_price(ELN_SPOT, ELN_STRIKE, ELN_TENOR,
                               ELN_RATE, ELN_VOL, "put", ELN_DIV_YIELD))

# Number of put units per notional
eln_put_units = ELN_NOTIONAL_HKD / ELN_STRIKE

# Fair coupon: risk-free interest + FV of put premium
eln_risk_free_interest = ELN_NOTIONAL_HKD * (np.exp(ELN_RATE * ELN_TENOR) - 1)
eln_put_premium_total = eln_put_units * eln_put_price
eln_coupon_cash = eln_risk_free_interest + eln_put_premium_total * np.exp(ELN_RATE * ELN_TENOR)
eln_coupon_rate_ann = (eln_coupon_cash / ELN_NOTIONAL_HKD) / ELN_TENOR

print("=" * 60)
print("ELN INDICATIVE TERM SHEET")
print("=" * 60)
print(f"  Underlying:           HSCEI (Hang Seng China Enterprises)")
print(f"  Spot:                 {ELN_SPOT:,.2f}")
print(f"  Strike:               {ELN_STRIKE:,.2f} ({ELN_STRIKE_PCT:.0%} of spot)")
print(f"  Tenor:                {ELN_TENOR:.2f}y (6 months)")
print(f"  HKD risk-free rate:   {ELN_RATE:.2%}")
print(f"  Dividend yield:       {ELN_DIV_YIELD:.2%}")
print(f"  Implied vol:          {ELN_VOL:.2%}")
print(f"  Embedded put value:   {eln_put_price:,.2f} index pts")
print(f"  Put units (per HKD {ELN_NOTIONAL_HKD:,.0f}): {eln_put_units:,.4f}")
print(f"  Total put premium:    HKD {eln_put_premium_total:,.2f}")
print(f"  Indicative coupon:    {eln_coupon_rate_ann:.2%} p.a.")
print(f"  Coupon cash (6M):     HKD {eln_coupon_cash:,.2f}")
print(f"  vs risk-free (6M):    HKD {eln_risk_free_interest:,.2f}")
print()

# --- Dealer greeks (dealer is LONG the put) ---
eln_greeks = bs_greeks(ELN_SPOT, ELN_STRIKE, ELN_TENOR,
                       ELN_RATE, ELN_VOL, "put", ELN_DIV_YIELD)

print("DEALER GREEKS (long put, per 1 index unit):")
for k, v in eln_greeks.items():
    print(f"  {k:12s}: {float(v):+.6f}")

print(f"\nSCALED TO {eln_put_units:,.2f} UNITS (HKD {ELN_NOTIONAL_HKD:,.0f} notional):")
print(f"  Delta (HKD): {float(eln_greeks['delta']) * eln_put_units:+,.2f}")
print(f"  Gamma:       {float(eln_greeks['gamma']) * eln_put_units:+,.6f}")
print(f"  Vega (1 vol pt): {float(eln_greeks['vega_1pct']) * eln_put_units:+,.2f}")
print(f"  Theta (per day): {float(eln_greeks['theta_day']) * eln_put_units:+,.2f}")

# --- Terminal payoff: three scenarios ---
spot_grid_eln = np.linspace(ELN_SPOT * 0.60, ELN_SPOT * 1.30, 500)

# Client terminal value
eln_terminal = (ELN_NOTIONAL_HKD + eln_coupon_cash
                - eln_put_units * np.maximum(ELN_STRIKE - spot_grid_eln, 0))

# Excess vs risk-free deposit
eln_risk_free_terminal = ELN_NOTIONAL_HKD * np.exp(ELN_RATE * ELN_TENOR)
eln_excess = eln_terminal - eln_risk_free_terminal

# Break-even: where excess = 0
eln_breakeven = ELN_STRIKE - (eln_coupon_cash - eln_risk_free_interest) / eln_put_units

# Three-scenario table
scenarios = pd.DataFrame([
    {"Scenario": "Rally (+15%)", "HSCEI at Maturity": ELN_SPOT * 1.15,
     "Client receives": f"HKD {ELN_NOTIONAL_HKD + eln_coupon_cash:,.0f}",
     "Excess vs deposit": f"HKD {eln_coupon_cash - eln_risk_free_interest:,.0f}"},
    {"Scenario": "Unchanged", "HSCEI at Maturity": ELN_SPOT,
     "Client receives": f"HKD {ELN_NOTIONAL_HKD + eln_coupon_cash:,.0f}",
     "Excess vs deposit": f"HKD {eln_coupon_cash - eln_risk_free_interest:,.0f}"},
    {"Scenario": "Sell-off (-20%)", "HSCEI at Maturity": ELN_SPOT * 0.80,
     "Client receives": f"HKD {ELN_NOTIONAL_HKD + eln_coupon_cash - eln_put_units * max(ELN_STRIKE - ELN_SPOT*0.80, 0):,.0f}",
     "Excess vs deposit": f"HKD {eln_coupon_cash - eln_risk_free_interest - eln_put_units * max(ELN_STRIKE - ELN_SPOT*0.80, 0):,.0f}"},
])
print("\nTHREE-SCENARIO ANALYSIS:")
display(scenarios)

# --- Payoff chart ---
# --- ELN client payoff chart ---
fig_eln = go.Figure()

fig_eln.add_trace(
    go.Scatter(
        x=spot_grid_eln,
        y=eln_excess,
        mode="lines",
        name="ELN excess vs HKD deposit",
        line=dict(width=3, color="#d62728"),
    )
)

# Reference line: no excess return versus plain HKD deposit
fig_eln.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray",
    line_width=1,
)

# Vertical reference lines
fig_eln.add_vline(
    x=eln_breakeven,
    line_dash="dot",
    line_color="red",
    line_width=2,
)

fig_eln.add_vline(
    x=ELN_STRIKE,
    line_dash="dot",
    line_color="black",
    line_width=2,
)

fig_eln.add_vline(
    x=ELN_SPOT,
    line_dash="dot",
    line_color="blue",
    line_width=2,
)

# Staggered labels
fig_eln.add_annotation(
    x=eln_breakeven,
    y=0.97,
    xref="x",
    yref="paper",
    text=f"Break-even<br>{eln_breakeven:,.0f}",
    showarrow=False,
    xanchor="right",
    xshift=-8,
    font=dict(color="red", size=13),
    bgcolor="rgba(255,255,255,0.85)",
)

fig_eln.add_annotation(
    x=ELN_STRIKE,
    y=0.84,
    xref="x",
    yref="paper",
    text=f"Strike<br>{ELN_STRIKE:,.0f}",
    showarrow=False,
    xanchor="left",
    xshift=8,
    font=dict(color="black", size=13),
    bgcolor="rgba(255,255,255,0.85)",
)

fig_eln.add_annotation(
    x=ELN_SPOT,
    y=0.97,
    xref="x",
    yref="paper",
    text=f"Spot<br>{ELN_SPOT:,.0f}",
    showarrow=False,
    xanchor="left",
    xshift=8,
    font=dict(color="blue", size=13),
    bgcolor="rgba(255,255,255,0.85)",
)

fig_eln.update_layout(
    title="6M HSCEI ELN — Client Excess Return vs HKD Deposit",
    xaxis_title="HSCEI at Maturity",
    yaxis_title="Excess Return (HKD)",
    template="plotly_white",
    height=550,
    margin=dict(t=100, l=90, r=50, b=80),
    showlegend=False,
)

fig_eln.show()

# --- Spot × Vol scenario grid ---
spot_shocks_eln = np.linspace(-20, 20, 21) / 100
vol_shocks_eln = np.linspace(-8, 8, 17) / 100

spot_scen_eln = ELN_SPOT * (1 + spot_shocks_eln)
vol_scen_eln = ELN_VOL + vol_shocks_eln
vol_scen_eln = vol_scen_eln[vol_scen_eln > 0]  # clip non-positive vol

S_m, V_m = np.meshgrid(spot_scen_eln, vol_scen_eln)
T_rem_eln = ELN_TENOR / 2

mtm_eln = bs_price(S_m, ELN_STRIKE, T_rem_eln,
                   ELN_RATE, V_m, "put", ELN_DIV_YIELD)
eln_mtm_pnl = (mtm_eln - eln_put_price) * eln_put_units

fig_eln_grid = go.Figure(data=go.Heatmap(
    x=np.round(spot_scen_eln, 0).astype(int),
    y=np.round(vol_scen_eln * 100, 1),
    z=eln_mtm_pnl,
    colorscale="RdYlGn", zmid=0,
    colorbar=dict(title="MTM P&L (HKD)")))
fig_eln_grid.update_layout(
    title="ELN Dealer Spot × Vol MTM P&L (3M into trade, HKD)",
    xaxis_title="HSCEI Spot",
    yaxis_title="Implied Vol (%)",
    template="plotly_white", height=550)
fig_eln_grid.show()


print(f"\nELN break-even HSCEI: {eln_breakeven:,.2f}")
print(f"ELN max loss: HSCEI → 0, client loses "
      f"HKD {eln_put_units * ELN_STRIKE - eln_coupon_cash:,.0f} net of coupon")


ELN INDICATIVE TERM SHEET
  Underlying:           HSCEI (Hang Seng China Enterprises)
  Spot:                 7,285.00
  Strike:               6,556.50 (90% of spot)
  Tenor:                0.50y (6 months)
  HKD risk-free rate:   3.85%
  Dividend yield:       3.40%
  Implied vol:          24.80%
  Embedded put value:   195.93 index pts
  Put units (per HKD 1,000,000): 152.5204
  Total put premium:    HKD 29,882.75
  Indicative coupon:    9.98% p.a.
  Coupon cash (6M):     HKD 49,900.04
  vs risk-free (6M):    HKD 19,436.48

DEALER GREEKS (long put, per 1 index unit):
  delta       : -0.237478
  gamma       : +0.000240
  vega_1pct   : +15.799216
  theta_day   : -1.031486

SCALED TO 152.52 UNITS (HKD 1,000,000 notional):
  Delta (HKD): -36.22
  Gamma:       +0.036617
  Vega (1 vol pt): +2,409.70
  Theta (per day): -157.32

THREE-SCENARIO ANALYSIS:


,Scenario,HSCEI at Maturity,Client receives,Excess vs deposit
0,Rally (+15%),"8,377.750000","HKD 1,049,900","HKD 30,464"
1,Unchanged,"7,285.000000","HKD 1,049,900","HKD 30,464"
2,Sell-off (-20%),"5,828.000000","HKD 938,789","HKD -80,648"



ELN break-even HSCEI: 6,356.77
ELN max loss: HSCEI → 0, client loses HKD 950,100 net of coupon


In [7]:
# =============================================================================
# DEALER VIEW — DISCRETE DELTA-HEDGE SIMULATION
# =============================================================================
# The dealer is LONG the embedded put. Hedge = short delta units of HSCEI.
# We simulate GBM paths and measure hedge P&L under different rebalancing
# frequencies and transaction costs.

def simulate_gbm(S0, drift, sigma, T, n_steps, n_paths, seed=42):
    """Generate GBM price paths."""
    dt = T / n_steps
    rng = np.random.default_rng(seed)
    Z = rng.standard_normal((n_paths, n_steps))
    log_ret = (drift - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    cum = np.column_stack([np.zeros(n_paths), np.cumsum(log_ret, axis=1)])
    return S0 * np.exp(cum)


def delta_hedge_put(paths, K, T, r, sigma, q=0.0,
                    rehedge_every=1, tc_bps=0.0):
    """Simulate discrete delta hedging of a long put position."""
    n_paths, n_dates = paths.shape
    n_steps = n_dates - 1
    dt = T / n_steps
    tc = tc_bps / 10_000

    # Initial: dealer is long put, hedges by going short delta shares
    premium = float(bs_price(paths[0, 0], K, T, r, sigma, "put", q))
    init_delta = bs_greeks(paths[:, 0], K, T, r, sigma, "put", q)["delta"]
    shares = np.copy(np.asarray(init_delta, dtype=float))  # negative for put
    # Cash = premium received - shares*S (shares < 0 so -shares*S > 0 = short proceeds)
    # minus transaction cost
    cash = premium - shares * paths[:, 0] - np.abs(shares * paths[:, 0]) * tc
    turnover = np.abs(shares * paths[:, 0])

    for step in range(1, n_steps + 1):
        cash *= np.exp(r * dt)
        if step < n_steps and step % rehedge_every == 0:
            remaining = T - step * dt
            if remaining <= 0:
                continue
            cur_S = paths[:, step]
            new_delta = bs_greeks(cur_S, K, remaining, r, sigma, "put", q)["delta"]
            trade = new_delta - shares
            trade_val = trade * cur_S
            cash -= trade_val
            cash -= np.abs(trade_val) * tc
            turnover += np.abs(trade_val)
            shares = np.copy(np.asarray(new_delta, dtype=float))

    # Terminal: unwind shares + receive put payoff
    terminal_S = paths[:, -1]
    put_payoff = np.maximum(K - terminal_S, 0.0)
    final_pnl = cash + shares * terminal_S + put_payoff

    return {"pnl": final_pnl, "turnover": turnover, "premium": premium}


# --- Run simulation ---
N_PATHS = 10_000
N_STEPS = int(TRADING_DAYS_PER_YEAR * ELN_TENOR)
DRIFT = ELN_RATE - ELN_DIV_YIELD

paths = simulate_gbm(ELN_SPOT, DRIFT, ELN_VOL, ELN_TENOR,
                     N_STEPS, N_PATHS, seed=42)

configs = {
    "Daily, 0 bps":    {"rehedge_every": 1,  "tc_bps": 0.0},
    "Weekly, 0 bps":   {"rehedge_every": 5,  "tc_bps": 0.0},
    "Monthly, 0 bps":  {"rehedge_every": 21, "tc_bps": 0.0},
    "Daily, 5 bps":    {"rehedge_every": 1,  "tc_bps": 5.0},
    "Daily, 10 bps cash-equity proxy": {"rehedge_every": 1, "tc_bps": 10.0,},
}

results = {}
for label, cfg in configs.items():
    results[label] = delta_hedge_put(
        paths, ELN_STRIKE, ELN_TENOR, ELN_RATE, ELN_VOL,
        q=ELN_DIV_YIELD, **cfg)

# --- Summary table ---
rows = []
for label, res in results.items():
    pnl = res["pnl"]
    rows.append({
        "Strategy": label,
        "Mean P&L": np.mean(pnl),
        "Std Dev": np.std(pnl, ddof=1),
        "5th %ile": np.percentile(pnl, 5),
        "95th %ile": np.percentile(pnl, 95),
        "Mean Turnover": np.mean(res["turnover"]),
    })

hedge_df = pd.DataFrame(rows)
display(hedge_df.style.format({
    "Mean P&L": "HKD {:,.2f}", "Std Dev": "HKD {:,.2f}",
    "5th %ile": "HKD {:,.2f}", "95th %ile": "HKD {:,.2f}",
    "Mean Turnover": "HKD {:,.0f}",
}))

# --- Histogram ---
# --- Histogram: selected hedge configurations ---
fig_hedge = go.Figure()

for label, color in [
    ("Daily, 0 bps", "#1f77b4"),
    ("Weekly, 0 bps", "#ff7f0e"),
    ("Daily, 10 bps cash-equity proxy", "#d62728"),
]:
    fig_hedge.add_trace(
        go.Histogram(
            x=results[label]["pnl"],
            name=label,
            opacity=0.55,
            nbinsx=80,
            marker_color=color,
        )
    )

fig_hedge.add_vline(
    x=0,
    line_dash="dash",
    line_color="black",
    line_width=1,
)

fig_hedge.update_layout(
    title="ELN Embedded Put — Dealer Delta-Hedge Residual P&L",
    xaxis_title="Hedged P&L at Maturity (HKD)",
    yaxis_title="Frequency",
    template="plotly_white",
    barmode="overlay",
    height=550,
)

fig_hedge.show()

print("\nKey observation: hedge P&L standard deviation increases with")
print("the rebalancing interval and transaction costs. The 10 bp")
print("cash-equity transaction-cost proxy reduces net hedge P&L.")

,Strategy,Mean P&L,Std Dev,5th %ile,95th %ile,Mean Turnover
0,"Daily, 0 bps",HKD 407.36,HKD 805.07,HKD -18.35,"HKD 2,342.30","HKD 22,049"
1,"Weekly, 0 bps",HKD 408.38,HKD 798.98,HKD -63.78,"HKD 2,319.09","HKD 10,976"
2,"Monthly, 0 bps",HKD 410.99,HKD 772.50,HKD -141.44,"HKD 2,218.83","HKD 5,575"
3,"Daily, 5 bps",HKD 396.22,HKD 804.63,HKD -29.54,"HKD 2,329.89","HKD 22,049"
4,"Daily, 10 bps cash-equity proxy",HKD 385.08,HKD 804.21,HKD -41.04,"HKD 2,317.87","HKD 22,049"



Key observation: hedge P&L standard deviation increases with
the rebalancing interval and transaction costs. The 10 bp
cash-equity transaction-cost proxy reduces net hedge P&L.


In [8]:
# =============================================================================
# SUITABILITY MATRIX — CLIENT OBJECTIVE → PRODUCT
# =============================================================================

suitability = pd.DataFrame([
    {"Client Objective": "Yield pickup on USD cash, natural CNH liability",
     "Structure": "6M USD/CNH DCI",
     "Embedded Option": "Short USD put / CNH call",
     "Market View": "Neutral-to-bullish USD/CNH",
     "Main Benefit": "Enhanced coupon vs USD deposit",
     "Main Risk": "Forced conversion to CNH at strike if USD weakens",
     "Typical Client": "HK corporate treasury with USD receivables"},

    {"Client Objective": "8–12% coupon, accept equity buffer",
     "Structure": "6M HSCEI ELN",
     "Embedded Option": "Short put at 90% of spot",
     "Market View": "Neutral-to-bullish HSCEI",
     "Main Benefit": "Above-market coupon",
     "Main Risk": "Equity loss below strike; no capital protection",
     "Typical Client": "Private-bank client, moderate risk appetite"},

    {"Client Objective": "Protect equity position, willing to cap upside",
     "Structure": "Collar (Appendix)",
     "Embedded Option": "Long put + short call",
     "Market View": "Uncertain, wants defined range",
     "Main Benefit": "Downside floor (put strike)",
     "Main Risk": "Capped upside, opportunity cost",
     "Typical Client": "Concentrated single-stock holder"},
])

print("CLIENT SUITABILITY MATRIX")
print("=" * 60)
display(suitability)

# =============================================================================
# FUNDING FOOTNOTE — OPTIONAL 50 BP ISSUER SPREAD
# =============================================================================
# In practice the note is issued on the bank's balance sheet. The bank
# funds at risk-free + spread. A 50 bp spread cheapens the note for the
# investor (lower coupon) or widens the desk margin.

FUNDING_SPREAD = 0.0050  # 50 bps

# DCI: funded coupon
dci_funded_coupon = dci_total_coupon - FUNDING_SPREAD
# ELN: funded coupon
eln_funded_rate = ELN_RATE + FUNDING_SPREAD
eln_funded_rf_interest = ELN_NOTIONAL_HKD * (np.exp(eln_funded_rate * ELN_TENOR) - 1)
# Put premium unchanged; coupon to client reduced
eln_funded_coupon_cash = eln_coupon_cash - (eln_funded_rf_interest - eln_risk_free_interest)
eln_funded_coupon_ann = (eln_funded_coupon_cash / ELN_NOTIONAL_HKD) / ELN_TENOR

print("\n" + "=" * 60)
print("FUNDING FOOTNOTE — IMPACT OF 50 BP ISSUER SPREAD")
print("=" * 60)
print(f"  DCI coupon (unfunded): {dci_total_coupon:.2%} → "
      f"(funded): {dci_funded_coupon:.2%} p.a.")
print(f"  ELN coupon (unfunded): {eln_coupon_rate_ann:.2%} → "
      f"(funded): {eln_funded_coupon_ann:.2%} p.a.")
print(f"  Spread captured by desk: 50 bps p.a. on notional")
print()
print("This is a simplified proxy for the XVA / treasury charge.")
print("A full implementation would require the issuer's CDS curve,")
print("a funding curve (SOFR + bank spread), and collateral assumptions.")
print("The Product & XVA team at SCB prices this properly;")
print("the 50 bp flat add-on here shows the direction of the adjustment.")

CLIENT SUITABILITY MATRIX


,Client Objective,Structure,Embedded Option,Market View,Main Benefit,Main Risk,Typical Client
0,"Yield pickup on USD cash, natural CNH liability",6M USD/CNH DCI,Short USD put / CNH call,Neutral-to-bullish USD/CNH,Enhanced coupon vs USD deposit,Forced conversion to CNH at strike if USD weakens,HK corporate treasury with USD receivables
1,"8–12% coupon, accept equity buffer",6M HSCEI ELN,Short put at 90% of spot,Neutral-to-bullish HSCEI,Above-market coupon,Equity loss below strike; no capital protection,"Private-bank client, moderate risk appetite"
2,"Protect equity position, willing to cap upside",Collar (Appendix),Long put + short call,"Uncertain, wants defined range",Downside floor (put strike),"Capped upside, opportunity cost",Concentrated single-stock holder



FUNDING FOOTNOTE — IMPACT OF 50 BP ISSUER SPREAD
  DCI coupon (unfunded): 6.18% → (funded): 5.68% p.a.
  ELN coupon (unfunded): 9.98% → (funded): 9.47% p.a.
  Spread captured by desk: 50 bps p.a. on notional

This is a simplified proxy for the XVA / treasury charge.
A full implementation would require the issuer's CDS curve,
a funding curve (SOFR + bank spread), and collateral assumptions.
The Product & XVA team at SCB prices this properly;
the 50 bp flat add-on here shows the direction of the adjustment.


---

## Model Limitations — What Was Omitted and Why

This section exists so that an interviewer can see that the author
knows the boundaries of the model.

### Things this model does correctly
- Garman-Kohlhagen / BSM pricing for European vanilla options
- Put-call parity verified to machine precision
- Greek sensitivities consistent with the pricing engine
- Discrete delta-hedge simulation with configurable frequency and costs
- Terminal payoff decomposition: coupon = option premium, not "free yield"

### Things this model does NOT do (and what it would take to add them)

| Omission | Impact | Effort to add |
|----------|--------|---------------|
| **Volatility smile / skew** | OTM puts are underpriced; real DCI/ELN coupons would differ | Local-vol or SABR calibration; ~2–4 weeks |
| **Issuer credit (CVA)** | Note price should reflect bank default probability | CDS curve + Monte Carlo; ~2 weeks |
| **Funding / FVA** | Desk funds at SOFR+spread, not risk-free | Funding curve integration; ~1 week |
| **Barriers / knock-in** | Many real ELNs have knock-in puts (cheaper) | Barrier option pricing (analytic or MC); ~1 week |
| **Stochastic rates** | USD and CNH rates modelled as constants | Hull-White or short-rate model; ~2 weeks |
| **Correlation (worst-of)** | Worst-of ELNs need joint simulation | Copula / multi-asset MC; ~3 weeks |
| **Early exercise / Bermudan** | Some DCIs are callable | Least-squares MC or PDE; ~2 weeks |

### What I would build next with more time
1. Calibrate a SABR smile to the USD/CNH 25-delta risk-reversal and
   butterfly, reprice the DCI, and show how much the coupon changes.
2. Add a knock-in barrier to the ELN put and compare the coupon to
   the vanilla version.
3. Build a simple issuer-credit adjustment using a flat hazard rate
   derived from the bank's 5-year CDS spread.
